<a href="https://colab.research.google.com/github/jeevithajeevitha3420-create/AI-Study-Companion-RAG/blob/main/AI_Study_Companion_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [85]:
!pip install -q transformers sentence-transformers faiss-cpu pypdf

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

print(os.listdir())

In [ ]:
from pypdf import PdfReader

pdf_name = list(uploaded.keys())[0]

reader = PdfReader(pdf_name)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

print("Number of pages:", len(reader.pages))
print("Characters extracted:", len(text))

In [ ]:
print(text[:2000])

In [ ]:
text

In [ ]:
chunk_size = 500
overlap = 50

chunks = []

start = 0

while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]

    chunks.append(chunk)

    start = end - overlap

print("Number of chunks:", len(chunks))

In [ ]:
print(chunks[0])

In [ ]:
print(chunks[1])

In [ ]:
print(chunks[2])

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

In [ ]:
embeddings = model.encode(chunks)

print("Embeddings created!")
print("Number of embeddings:", len(embeddings))
print("Embedding size:", len(embeddings[0]))

In [ ]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS database created!")
print("Total chunks stored:", index.ntotal)

In [ ]:
question = "What is machine learning?"

question_embedding = model.encode([question])

question_embedding = np.array(question_embedding).astype("float32")

distances, indices = index.search(question_embedding, 3)

print("Relevant chunks:")
print(indices)

In [ ]:
for i in indices[0]:
    print("----------")
    print(chunks[i])

In [ ]:
!pip install -q transformers accelerate

In [ ]:
!pip install -q -U transformers sentencepiece accelerate

In [ ]:
!pip install -q torch==2.11.0 torchvision==0.26.0

In [ ]:
import torch

print("PyTorch version:", torch.__version__)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded successfully!")

In [ ]:
question = "What is machine learning?"

inputs = tokenizer(
    question,
    return_tensors="pt"
)

outputs = llm.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

In [ ]:
def ask_rag(question, k=3):

    # 1. Convert question into embedding
    question_embedding = model.encode([question])
    question_embedding = np.array(question_embedding).astype("float32")

    # 2. Search FAISS
    distances, indices = index.search(question_embedding, k)

    # 3. Get relevant chunks
    retrieved_chunks = []

    for i in indices[0]:
        retrieved_chunks.append(chunks[i])

    # 4. Combine retrieved information
    context = "\n\n".join(retrieved_chunks)

    # 5. Create prompt
    prompt = f"""
Use the following study notes to answer the question.

STUDY NOTES:
{context}

QUESTION:
{question}

Give a simple and clear answer based only on the study notes.
"""

    # 6. Tokenize prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # 7. Generate answer
    outputs = llm.generate(
        **inputs,
        max_new_tokens=150
    )

    # 8. Convert output to text
    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, retrieved_chunks

In [ ]:
answer, sources = ask_rag(
    "What is machine learning?"
)

print("🤖 ANSWER:")
print(answer)

In [ ]:
print("\n📚 SOURCES:")

for i, source in enumerate(sources):
    print(f"\n--- Source {i+1} ---")
    print(source)

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_name)

chunks = []
chunk_sources = []

chunk_size = 500
overlap = 50

for page_number, page in enumerate(reader.pages, start=1):

    page_text = page.extract_text()

    if not page_text:
        continue

    start = 0

    while start < len(page_text):

        end = start + chunk_size

        chunk = page_text[start:end]

        chunks.append(chunk)
        chunk_sources.append(page_number)

        start = end - overlap

print("Total chunks:", len(chunks))
print("Total sources:", len(chunk_sources))

In [ ]:
embeddings = model.encode(chunks)

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS updated!")
print("Total chunks:", index.ntotal)

In [ ]:
def ask_rag(question, k=3):

    question_embedding = model.encode([question])
    question_embedding = np.array(question_embedding).astype("float32")

    distances, indices = index.search(
        question_embedding,
        k
    )

    retrieved_chunks = []
    retrieved_pages = []

    for i in indices[0]:

        retrieved_chunks.append(chunks[i])
        retrieved_pages.append(chunk_sources[i])

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
Use the following study notes to answer the question.

STUDY NOTES:
{context}

QUESTION:
{question}

Give a simple and clear answer based only on the study notes.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer, retrieved_chunks, retrieved_pages

In [ ]:
answer, sources, pages = ask_rag(
    "What is machine learning?"
)

print("🤖 ANSWER:")
print(answer)

print("\n📌 SOURCES:")

for page in pages:
    print("Page:", page)

In [ ]:
def generate_quiz(topic, k=5):

    # Retrieve relevant chunks
    question_embedding = model.encode([topic])
    question_embedding = np.array(question_embedding).astype("float32")

    distances, indices = index.search(
        question_embedding,
        k
    )

    retrieved_chunks = []

    for i in indices[0]:
        retrieved_chunks.append(chunks[i])

    context = "\n\n".join(retrieved_chunks)

    # Prompt for quiz generation
    prompt = f"""
Create 5 multiple-choice questions from the study notes below.

STUDY NOTES:
{context}

For each question:
1. Give the question.
2. Give four options: A, B, C, D.
3. Give the correct answer.

Keep the questions simple and based only on the study notes.
"""

    # Generate quiz
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=500
    )

    quiz = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return quiz

In [ ]:
quiz = generate_quiz("machine learning")

print(quiz)

In [ ]:
def get_quiz_context(topic, k=5):

    question_embedding = model.encode([topic])
    question_embedding = np.array(question_embedding).astype("float32")

    distances, indices = index.search(
        question_embedding,
        k
    )

    retrieved_chunks = []

    for i in indices[0]:
        retrieved_chunks.append(chunks[i])

    return "\n\n".join(retrieved_chunks)

In [ ]:
context = get_quiz_context("machine learning")

prompt = f"""
Based only on these study notes:

{context}

Create ONE multiple choice question.

Format exactly like this:

QUESTION: ...
A: ...
B: ...
C: ...
D: ...
ANSWER: A/B/C/D
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = llm.generate(
    **inputs,
    max_new_tokens=150
)

quiz_question = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(quiz_question)

In [ ]:
answer = input("Enter your answer (A/B/C/D): ").upper()

correct_answer = "B"

if answer == correct_answer:
    print("✅ Correct!")
else:
    print("❌ Incorrect!")
    print("Correct answer:", correct_answer)

In [ ]:
correct_answer = "B"

In [ ]:
def create_quiz_question(topic):

    context = get_quiz_context(topic)

    prompt = f"""
Based ONLY on the study notes below, create ONE multiple-choice question.

Study notes:
{context}

Use EXACTLY this format:

QUESTION: your question
A: option A
B: option B
C: option C
D: option D
ANSWER: A

The ANSWER must be only A, B, C, or D.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=180
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [ ]:
quiz = create_quiz_question("machine learning")

print(quiz)

In [ ]:
def find_correct_answer(question, options, topic):

    context = get_quiz_context(topic)

    options_text = ""

    for i, option in enumerate(options):
        options_text += f"{chr(65+i)}: {option}\n"

    prompt = f"""
Study notes:
{context}

Question:
{question}

Options:
{options_text}

Which option is correct?

Reply with ONLY one letter:
A
B
C
or
D
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=5
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip().upper()

    match = re.search(r"\b([ABCD])\b", answer)

    if match:
        return match.group(1)

    return None

In [ ]:
quiz_data = [
    {
        "question": "What is machine learning?",
        "options": [
            "A method where computers learn from data",
            "A type of computer hardware",
            "A programming language",
            "A database system"
        ],
        "answer": "A"
    },

    {
        "question": "Which type of learning uses labeled data?",
        "options": [
            "Unsupervised learning",
            "Supervised learning",
            "Reinforcement learning",
            "Random learning"
        ],
        "answer": "B"
    },

    {
        "question": "Which algorithm is commonly used for classification?",
        "options": [
            "Linear Regression",
            "K-Means",
            "Decision Tree",
            "PCA"
        ],
        "answer": "C"
    }
]

In [ ]:
import random

quiz = random.choice(quiz_data)

print("QUESTION:")
print(quiz["question"])

print()

for i, option in enumerate(quiz["options"]):
    print(f"{chr(65+i)}: {option}")

print("\nCorrect answer:", quiz["answer"])

In [ ]:
def get_quiz_material(topic, k=5):

    question_embedding = model.encode([topic])
    question_embedding = np.array(question_embedding).astype("float32")

    distances, indices = index.search(
        question_embedding,
        k
    )

    material = []

    for i in indices[0]:
        material.append(chunks[i])

    return "\n\n".join(material)

In [ ]:
material = get_quiz_material("machine learning")

print(material)

In [ ]:
def generate_question_from_notes(topic):

    material = get_quiz_material(topic)

    prompt = f"""
Read these study notes:

{material}

Create ONE multiple-choice question from these notes.

Return ONLY the question.
Do not give options.
Do not give the answer.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=60
    )

    question = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return question.strip()

In [ ]:
question = generate_question_from_notes("machine learning")

print("QUESTION:")
print(question)

In [ ]:
def generate_options_from_notes(question, topic):

    material = get_quiz_material(topic)

    prompt = f"""
Study notes:
{material}

Question:
{question}

Create exactly 4 short answer options for this question.

Return ONLY the four options, one per line.
Do not write A, B, C, or D.
Do not give the correct answer.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.8
    )

    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    options = [
        line.strip()
        for line in result.split("\n")
        if line.strip()
    ]

    return options[:4]


In [ ]:
question = generate_question_from_notes("machine learning")

options = generate_options_from_notes(
    question,
    "machine learning"
)

print("QUESTION:")
print(question)

print("\nOPTIONS:")

for i, option in enumerate(options):
    print(f"{chr(65+i)}: {option}")

In [ ]:
material = get_quiz_material("V-model")

print(material[:3000])

In [ ]:
import random

quiz_data = [
    {
        "question": "The V-model is a variation of which model?",
        "options": [
            "Waterfall model",
            "Spiral model",
            "Agile model",
            "Prototype model"
        ],
        "answer": "Waterfall model"
    },

    {
        "question": "Which model is represented in a V shape?",
        "options": [
            "V-model",
            "Spiral model",
            "Incremental model",
            "Prototype model"
        ],
        "answer": "V-model"
    }
]

In [ ]:
quiz = random.choice(quiz_data)

options = quiz["options"].copy()

random.shuffle(options)

letters = ["A", "B", "C", "D"]

print("QUESTION:")
print(quiz["question"])

print("\nOPTIONS:")

for letter, option in zip(letters, options):
    print(f"{letter}: {option}")

correct_letter = letters[options.index(quiz["answer"])]

print("\nCorrect answer:", correct_letter)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

score = 0

quiz = random.choice(quiz_data)

options = quiz["options"].copy()
random.shuffle(options)

letters = ["A", "B", "C", "D"]

print("QUESTION:")
print(quiz["question"])

print()

for letter, option in zip(letters, options):
    print(f"{letter}: {option}")

correct_letter = letters[options.index(quiz["answer"])]

print("\nChoose your answer:")

buttons = []

for letter in letters:

    button = widgets.Button(
        description=letter,
        layout=widgets.Layout(width="80px")
    )

    buttons.append(button)


output = widgets.Output()


def check_answer(button):

    with output:

        clear_output()

        student_answer = button.description

        print("Your answer:", student_answer)

        if student_answer == correct_letter:
            print("✅ Correct!")
            print("Score: 1 / 1")
        else:
            print("❌ Incorrect!")
            print("Correct answer:", correct_letter)
            print("Score: 0 / 1")


for button in buttons:
    button.on_click(check_answer)


display(widgets.HBox(buttons))
display(output)

In [ ]:
import random
import ipywidgets as widgets
from IPython.display import display, clear_output

score = 0
current_question = 0
total_questions = 5

output = widgets.Output()

def show_question():

    global current_question

    if current_question >= total_questions:
        with output:
            clear_output()
            print("🎯 QUIZ FINISHED!")
            print()
            print("Final Score:", score, "/", total_questions)
            print("Percentage:", round((score / total_questions) * 100, 1), "%")
        return

    quiz = random.choice(quiz_data)

    options = quiz["options"].copy()
    random.shuffle(options)

    letters = ["A", "B", "C", "D"]

    correct_letter = letters[
        options.index(quiz["answer"])
    ]

    with output:
        clear_output()

        print("=" * 50)
        print("Question", current_question + 1, "of", total_questions)
        print()
        print(quiz["question"])
        print()

        for letter, option in zip(letters, options):
            print(f"{letter}: {option}")

        print()
        print("Choose your answer:")

    buttons = []

    for letter in letters:

        button = widgets.Button(
            description=letter,
            layout=widgets.Layout(width="80px")
        )

        buttons.append(button)

        def check_answer(button, correct_letter=correct_letter):

            global score, current_question

            with output:
                clear_output()

                student_answer = button.description

                print("Your answer:", student_answer)
                print()

                if student_answer == correct_letter:
                    print("✅ Correct!")
                    score += 1
                else:
                    print("❌ Incorrect!")
                    print("Correct answer:", correct_letter)

                current_question += 1

                print()
                print("Current score:", score)

            next_button = widgets.Button(
                description="Next Question",
                button_style="primary"
            )

            def next_question(b):
                next_button.close()
                for btn in buttons:
                    btn.close()
                show_question()

            next_button.on_click(next_question)

            display(next_button)

        button.on_click(check_answer)

    display(widgets.HBox(buttons))


display(output)

show_question()